# Step 3: Simulating an Intervention & Statistical Testing

**What we're doing:** The Telco dataset is observational — nobody actually
ran a retention-offer experiment. So we SIMULATE one: we randomly assign a
synthetic 'retention offer' to a subset of high-risk customers, simulate a
plausible effect on their churn outcome, then test for it statistically as
if it were real experimental data.

**Why this matters / what it proves:** This is the difference between
'I can describe data' (EDA) and 'I can determine whether an observed
difference is real or just noise' (inferential statistics). This is
exactly the skill behind A/B testing, which almost every data role expects.

**Important honesty note for your README:** Because we SIMULATE the
intervention and its effect size ourselves, this section demonstrates
the *statistical methodology* correctly — not a real causal discovery
about Telco customers. Be explicit about this distinction when you write
it up; it shows scientific integrity, which interviewers notice.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/churn-platform'

import pandas as pd
import numpy as np
from scipy import stats

np.random.seed(42)  # reproducibility — always seed simulations

df = pd.read_csv(f'{PROJECT_ROOT}/data/processed/churn_clean.csv')
df.shape

Mounted at /content/drive


(7043, 21)

## 3.1 Design the simulated experiment

**Setup:** Take the highest-risk segment from our EDA (month-to-month
contract customers). Randomly split them into a Treatment group (gets a
synthetic 'retention offer') and a Control group (gets nothing).

**Why random assignment specifically:** Random assignment is THE thing
that makes an A/B test valid. It means any difference we see afterward
between groups is either (a) the treatment effect, or (b) random chance —
and NOT some pre-existing difference between the groups (like one group
happening to have more loyal customers already). This is the core idea
of a randomized controlled trial.

In [2]:
at_risk = df[df['Contract'] == 'Month-to-month'].copy()
print(f'At-risk population size: {len(at_risk)}')

# Random 50/50 assignment to treatment/control
at_risk['group'] = np.random.choice(['treatment', 'control'], size=len(at_risk), p=[0.5, 0.5])
print(at_risk['group'].value_counts())

At-risk population size: 3875
group
control      1943
treatment    1932
Name: count, dtype: int64


In [3]:
# Simulate the effect: assume the retention offer reduces churn probability
# by an absolute 8 percentage points for treated customers who were going
# to churn. This effect size is an ASSUMPTION we're injecting — change it
# to see how it affects statistical power later.

TREATMENT_EFFECT = 0.08  # 8 percentage point reduction in churn probability

def simulate_outcome(row):
    base_churn_prob = 0.42  # observed month-to-month churn rate, roughly
    if row['group'] == 'treatment':
        churn_prob = max(base_churn_prob - TREATMENT_EFFECT, 0)
    else:
        churn_prob = base_churn_prob
    return np.random.binomial(1, churn_prob)

at_risk['simulated_churn'] = at_risk.apply(simulate_outcome, axis=1)

summary = at_risk.groupby('group')['simulated_churn'].agg(['mean', 'count'])
summary

,mean,count
group,,
control,0.413793,1943
treatment,0.317805,1932


## 3.2 Hypothesis test: chi-square test of independence

**Why chi-square here:** Both our variables — `group` (treatment/control)
and `simulated_churn` (yes/no) — are categorical. Chi-square tests whether
two categorical variables are independent of each other. If group and
churn outcome are truly independent (the offer does nothing), we'd expect
the churn rate in each group to be roughly equal.

**The hypotheses, stated explicitly:**
- H0 (null): churn rate is the same in treatment and control (offer has no effect)
- H1 (alternative): churn rate differs between treatment and control

**How to read the p-value:** the p-value is the probability of seeing a
difference at least this large BY RANDOM CHANCE ALONE, if H0 were true.
A small p-value (conventionally < 0.05) means this result would be
unlikely under pure chance, so we reject H0 in favor of H1.

In [4]:
contingency_table = pd.crosstab(at_risk['group'], at_risk['simulated_churn'])
print('Contingency table (rows=group, cols=churned 0/1):')
print(contingency_table)
print()

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f'Chi-square statistic: {chi2:.4f}')
print(f'Degrees of freedom: {dof}')
print(f'p-value: {p_value:.6f}')

alpha = 0.05
if p_value < alpha:
    print(f'\np-value < {alpha}: reject H0 — the retention offer effect is statistically significant.')
else:
    print(f'\np-value >= {alpha}: fail to reject H0 — not enough evidence the offer has an effect.')

Contingency table (rows=group, cols=churned 0/1):
simulated_churn     0    1
group                     
control          1139  804
treatment        1318  614

Chi-square statistic: 38.0556
Degrees of freedom: 1
p-value: 0.000000

p-value < 0.05: reject H0 — the retention offer effect is statistically significant.


## 3.3 Confidence interval on the difference in churn rates

**Why we also want a confidence interval, not just a p-value:** a p-value
only tells you IF there's likely an effect. A confidence interval tells you
the plausible RANGE of how big that effect is — which is what a business
stakeholder actually needs to decide whether the offer is worth the cost.
'Statistically significant' doesn't mean 'big enough to matter'; the CI
lets you check both at once.

In [5]:
treat = at_risk[at_risk['group'] == 'treatment']['simulated_churn']
ctrl = at_risk[at_risk['group'] == 'control']['simulated_churn']

p1, n1 = treat.mean(), len(treat)
p2, n2 = ctrl.mean(), len(ctrl)
diff = p1 - p2

# Standard error for the difference between two proportions
se = np.sqrt((p1 * (1 - p1) / n1) + (p2 * (1 - p2) / n2))
z = 1.96  # 95% confidence
ci_low, ci_high = diff - z * se, diff + z * se

print(f'Treatment churn rate: {p1:.4f} (n={n1})')
print(f'Control churn rate:   {p2:.4f} (n={n2})')
print(f'Observed difference:  {diff:.4f}')
print(f'95% CI for the difference: [{ci_low:.4f}, {ci_high:.4f}]')

# How to read this: we're 95% confident the TRUE effect of the offer
# (in this simulated world) lies between ci_low and ci_high. If this
# interval does NOT include 0, that's consistent with our chi-square
# result above (a real, non-zero effect).

Treatment churn rate: 0.3178 (n=1932)
Control churn rate:   0.4138 (n=1943)
Observed difference:  -0.0960
95% CI for the difference: [-0.1262, -0.0658]


## 3.4 t-test variant (for completeness)

The chi-square test above treats churn as categorical. An independent
samples t-test on the same 0/1 outcome gives an equivalent answer (a 0/1
variable's mean IS a proportion) — included here so you can show fluency
with both tests and explain when each is more natural
(t-test → comparing means of a continuous/binary variable;
chi-square → comparing distributions across categorical levels).

In [6]:
t_stat, t_pvalue = stats.ttest_ind(treat, ctrl)
print(f't-statistic: {t_stat:.4f}')
print(f'p-value: {t_pvalue:.6f}')

# Also worth testing on a genuinely continuous variable, e.g. does the
# treatment group's MonthlyCharges differ from control (should NOT, since
# assignment was random — this acts as a 'balance check', standard
# practice in real A/B test reporting to confirm randomization worked).
balance_t, balance_p = stats.ttest_ind(
    at_risk[at_risk['group']=='treatment']['MonthlyCharges'],
    at_risk[at_risk['group']=='control']['MonthlyCharges']
)
print(f'\nBalance check (MonthlyCharges, should be non-significant): p = {balance_p:.4f}')

t-statistic: -6.2317
p-value: 0.000000

Balance check (MonthlyCharges, should be non-significant): p = 0.1382


## 3.5 Write-up template for the README

> We simulated a retention-offer A/B test on month-to-month customers
> (n=X), randomly assigning a synthetic offer that we modeled as reducing
> churn probability by 8 percentage points. A chi-square test of
> independence found this difference statistically significant
> (chi2=X, p=X), and the 95% confidence interval for the effect size was
> [X, X]. Note: this is a simulated intervention used to demonstrate
> experimental design and hypothesis testing methodology, not a finding
> about real Telco customer behavior.